#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType
import uuid
from datetime import datetime

In [0]:
# crm_cust_info_raw time (starts here...)

start_time = datetime.now()

# Reading From Bronze

In [0]:
df = spark.table("salesdatalakehouse.bronze.crm_cust_info_raw")

# Data Transformations

## Trimming

In [0]:
for col_name, col_type in df.dtypes:
    if col_type == "string":
        df = df.withColumn(col_name, trim(df[col_name]))

## Normalization

In [0]:
df = (
        df
        .withColumn(
                "cst_gndr",
                F.when(F.upper(col("cst_gndr")) == 'F', 'Female')
                 .when(F.upper(col("cst_gndr")) == 'M', 'Male')
                 .otherwise('n/a')
                )
        .withColumn(
                "cst_marital_status",
                F.when(F.upper(col("cst_marital_status")) == 'S', 'Single')
                 .when(F.upper(col("cst_marital_status")) == 'M', 'Married')
                 .otherwise('n/a')
                )
     )

## Drop Invalid Customer IDs (all columns contain nulls)

In [0]:
df = df.filter(~col("cst_key").isin(['13451235', 'PO25', 'A01Ass', 'SF566']))

## Renaming Colmns

In [0]:
rename_map = {
    "cst_id"             : "customer_id",
    "cst_key"            : "customer_key",
    "cst_firstname"      : "first_name",
    "cst_lastname"       : "last_name",
    "cst_marital_status" : "marital_status",
    "cst_gndr"           : "gender",
    "cst_create_date"    : "create_date"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

# Load Into Silver

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable('salesdatalakehouse.silver.crm_cust_info')

# crm_cust_info Logging

In [0]:
# crm_cust_info_raw time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Silver'
schema_name = 'silver'
table_name = 'crm_cust_info'
row_inserted = spark.sql('''SELECT COUNT(*) FROM salesdatalakehouse.silver.crm_cust_info''').collect()[0][0]

end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ crm_cust_info info Logged with run {run_id}")